# Biological BMI Paper — Longitudinal Change of Biological BMI during the Arivale Program

***by Kengo Watanabe***  

In this notebook, longitudinal changes of BMI and biological BMIs are assessed with linear mixed model (LMM), which is same with GLMM with Gaussian family with an identity link function, because log-scaled BMI and biological BMIs are sufficiently symmetric distribution.  

Input:  
* Time-series MetBMI predictions: 210106_Biological-BMI-paper_longitudinal-LASSO_times-series-metBMI-FemaleMale.csv  
* Time-series ProtBMI predictions: 210106_Biological-BMI-paper_longitudinal-LASSO_times-series-protBMI-FemaleMale.csv  
* Time-series ChemBMI predictions: 210106_Biological-BMI-paper_longitudinal-LASSO_times-series-chemBMI-FemaleMale.csv  
* Cleaned time-series BMI data: 210104_Biological-BMI-paper_data-cleaning-BMI-omics_time-series-bmiDF-without-imputation_final-cohort.tsv  
* Cleaned time-series combined omics (measurement count): 210104_Biological-BMI-paper_data-cleaning-BMI-omics_time-series-combiDF-without-imputation_final-cohort.tsv  

Output:  
* Figure 5  

Original notebook:  
* 210823_Biological-BMI-paper_longitudinal-LMM-ver3(figure-layout)  

In [ ]:
import numpy as np
import pandas as pd
import scipy.stats as stats
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
#For Arial font
#!conda install -c conda-forge -y mscorefonts
import warnings
warnings.filterwarnings('ignore')
from IPython.display import display
import time

from sklearn.preprocessing import StandardScaler
import statsmodels.api as sm
from statsmodels.stats import multitest as multi
import matplotlib.lines as mlines

!conda list

## 1. Metabolomics

### 1-1. Prepare the time-series DF

In [ ]:
#Import time-series biological BMI
print('Sex-stratified model')
fileDir = './ExportData/'
fileName = '210106_Biological-BMI-paper_longitudinal-LASSO_times-series-metBMI-FemaleMale.csv'
tsDF_FM = pd.read_csv(fileDir+fileName, dtype={'public_client_id': str}).set_index('KeyIndex')
display(tsDF_FM)

### 1-2. Prepare DFs for LMM with the linear regresion spline for time

In [ ]:
#Check measurement distribution
sns.set(style='ticks', font='Arial', context='notebook')
plt.figure(figsize=(4, 3))
sns.distplot(tsDF_FM['days_in_program'], color='b')
sns.despine()
plt.axvline(x=365.25/12*0, **{'linestyle':'--', 'color':'gray'})
plt.axvline(x=365.25/12*6, **{'linestyle':'--', 'color':'gray'})
plt.axvline(x=365.25/12*12, **{'linestyle':'--', 'color':'gray'})
plt.axvline(x=365.25/12*18, **{'linestyle':'--', 'color':'gray'})
plt.axvspan(xmin=365.25/12*18, xmax=plt.gca().get_xlim()[1], facecolor='gray', alpha=0.2)
plt.ylabel('Density')
plt.xlabel('Days in program')
plt.show()

In [ ]:
#Sex-stratified model
print('Sex-stratified model')

#Drop unnecessary variables
tempDF = tsDF_FM[['public_client_id', 'days_in_program', 'Season', 'MetBMI', 'BaseMetBMI',
                  'BaseMetBMI_class', 'BaseBMI_class', 'Sex', 'BaseAge', 'PC1', 'PC2', 'PC3', 'PC4', 'PC5']]
print('Time-series DF nrows before elimination:', len(tempDF))
print(' -> Unique ID:', len(tempDF['public_client_id'].unique()))

#Select measurements used in the longitudinal analysis
month_threshold = 18
tempDF = tempDF.loc[tempDF['days_in_program'] <= 365.25/12*month_threshold]
print('Time-series DF nrows after selecting measurements during the taget period:', len(tempDF))
print(' -> Unique ID:', len(tempDF['public_client_id'].unique()))

#Select the cohort used in the longitudinal analysis
##Select participants who have more than 2 measuremnts in BMI
fileDir = './ExportData/'
fileName = '210104_Biological-BMI-paper_data-cleaning-BMI-omics_time-series-bmiDF-without-imputation_final-cohort.tsv'
tempDF1 = pd.read_csv(fileDir+fileName, sep='\t', dtype={'public_client_id': str}).set_index('public_client_id')
tempDF1 = tempDF1.loc[tempDF1['days_in_program'] <= 365.25/12*month_threshold]
tempS = tempDF1.index.value_counts()
tempL1 = tempS.loc[tempS>=2].index.tolist()
##Select participants who have more than 2 measuremnts in omics
fileDir = './ExportData/'
fileName = '210104_Biological-BMI-paper_data-cleaning-BMI-omics_time-series-combiDF-without-imputation_final-cohort.tsv'
tempDF1 = pd.read_csv(fileDir+fileName, sep='\t', dtype={'public_client_id': str}).set_index('public_client_id')
tempDF1 = tempDF1.loc[tempDF1['days_in_program'] <= 365.25/12*month_threshold]
tempS = tempDF1.index.value_counts()
tempL2 = tempS.loc[tempS>=2].index.tolist()
##Select participants who have more than 2 measuremnts in both BMI and omics
tempL = list(set(tempL1) & set(tempL2))
print('Participants who have more than 2 measurements in both BMI and omics:', len(tempL))
tempDF = tempDF.loc[tempDF['public_client_id'].isin(tempL)]
print('Time-series DF nrows after selecting participants who have ≥2 measuremnts in both BMI and omics:',
      len(tempDF))
print(' -> Unique ID:', len(tempDF['public_client_id'].unique()))

#Select the participants who have all covariates
tempDF = tempDF.dropna()
print('Time-series DF nrows after selecting participants who have all covariates:', len(tempDF))
print(' -> Unique ID:', len(tempDF['public_client_id'].unique()))

#Check measurement distribution
sns.set(style='ticks', font='Arial', context='talk')
plt.figure(figsize=(4, 3))
sns.distplot(tempDF['days_in_program'], color='b')
sns.despine()
plt.axvline(x=365.25/12*0, **{'linestyle':'--', 'color':'gray'})
plt.axvline(x=365.25/12*6, **{'linestyle':'--', 'color':'gray'})
plt.axvline(x=365.25/12*12, **{'linestyle':'--', 'color':'gray'})
plt.axvline(x=365.25/12*18, **{'linestyle':'--', 'color':'gray'})
plt.axvspan(xmin=365.25/12*18, xmax=plt.gca().get_xlim()[1], facecolor='gray', alpha=0.2)
plt.ylabel('Density')
plt.xlabel('Days in program')
plt.show()

#Calculate rate of change
tempDF['MetBMIchange'] = (tempDF['MetBMI'] - tempDF['BaseMetBMI']) / tempDF['BaseMetBMI'] * 100

#Check distribution
print('Skewness of rate of change:', stats.skew(tempDF['MetBMIchange']))
sns.set(style='ticks', font='Arial', context='notebook')
plt.figure(figsize=(4, 3))
sns.distplot(tempDF['MetBMIchange'], color='b')
sns.despine()
plt.ylabel('Density')
plt.xlabel('Rate of change [%]')
plt.show()

#Add dummy variables for the linear regression spline for time
nknots = 3
tempL = [365.25/12 * month_threshold/nknots * (knot+1) for knot in range(nknots)]#Days of knots
tempL1 = []#2nd piece
tempL2 = []#3rd piece
for days in tempDF['days_in_program'].tolist():
    if days < tempL[0]:#[0 month, 6 months)
        tempL1.append(0.0)
        tempL2.append(0.0)
    elif days < tempL[1]:#[6 months, 12 months)
        tempL1.append(days - tempL[0])
        tempL2.append(0.0)
    elif days <= tempL[2]:#[12 months, 18 months]
        tempL1.append(days - tempL[0])
        tempL2.append(days - tempL[1])
tempDF['DaysP2'] = tempL1
tempDF['DaysP3'] = tempL2

#Sort to make 'Normal' the standard in the baseline BMI-stratified LMM (Normal < Obese < Overweight)
tempDF = tempDF.sort_values(by=['BaseBMI_class'], ascending=True)

#Standardize numeric independent variables
scaler_FM = StandardScaler(copy=True, with_mean=True, with_std=True)#Use later again during predictions
tempL = ['days_in_program', 'DaysP2', 'DaysP3', 'BaseAge', 'PC1', 'PC2', 'PC3', 'PC4', 'PC5']
scaler_FM.fit(tempDF[tempL])
tempDF1 = pd.DataFrame(scaler_FM.transform(tempDF[tempL]), index=tempDF.index,
                       columns=['DaysZ', 'DaysP2Z', 'DaysP3Z', 'BaseAgeZ', 'PC1Z', 'PC2Z', 'PC3Z', 'PC4Z', 'PC5Z'])
tempDF = pd.merge(tempDF, tempDF1, left_index=True, right_index=True, how='inner')

#Confirm examples
sns.set(style='ticks', font='Arial', context='notebook')
plt.figure(figsize=(4, 3))
sns.distplot(tempDF['DaysZ'], label='DaysZ')
sns.distplot(tempDF['BaseAgeZ'], label='BaseAgeZ')
sns.distplot(tempDF['PC1Z'], label='PC1Z')
sns.despine()
plt.ylabel('Density')
plt.xlabel('Z-score')
tempL = [365.25/12 * month_threshold/nknots * (knot+1) for knot in range(nknots)]#Days of knots
tempS = (pd.Series(tempL) - scaler_FM.mean_[0]) / scaler_FM.scale_[0]#Z-score transformation
for knot in tempS:
    plt.axvline(x=knot, **{'linestyle':'--', 'color':'gray'})
plt.legend(bbox_to_anchor=(1, 0.5), loc='center left', borderaxespad=1)
plt.show()

#Update
tsDF_FM = tempDF
display(tsDF_FM)

### 1-3. LMM with random intercepts and random slopes for time

> Days in the program is fitted as a numeric variable using linear regression splines (cf. Zubair, N. et al. Sci. Rep. 2019).  
> ***–> Allows differences in the trajectory of biological BMI change.***
>
> Random intercepts and random slopes for days in the program are used as random effects (cf. Zubair, N. et al. Sci. Rep. 2019).  
> ***–> Allows individual differences in the intercept and the rate of change of biological BMI over the program.***  
> –> Random slope is applied not for the piecewise variable of time but only for the overall variable of time; otherwise, leads to "LinAlgError: Singular matrix" due to small sample size per most individuals. Also, it is not so weird to assume the random slope is consistent during the program.  
>
> ***Eliminate "Underweight" class in the baseline obesity class-stratified models.***  
> <– Otherwise, leads to "LinAlgError: Singular matrix" due to small sample population.  

In [ ]:
#Sex-stratified model-predicted biological BMI
print('Sex-stratified model-predicted biological BMI')

#Overall LMM adjusted for general covariates
fe_formula = 'MetBMIchange ~ DaysZ + DaysP2Z + DaysP3Z'\
    ' + C(Sex) + BaseAgeZ + C(Season) + PC1Z + PC2Z + PC3Z + PC4Z + PC5Z'
model = sm.MixedLM.from_formula(fe_formula, groups='public_client_id', data=tsDF_FM,
                                re_formula='DaysZ')#Random intercepts for each group are included as default
t_start = time.time()
LMM1_FM = model.fit(method=['bfgs', 'lbfgs', 'cg', 'powell', 'nm'])#Back up for failure of convergence
t_elapsed = time.time() - t_start
print('\n• Model 1: LMM adjusted for general covariates')
display(LMM1_FM.summary())
print('(Note that the estimates for fixed effects and the variances for random effects are shown in the single table.)')
##Visualize b-coefs and g-coefs
tempDF = pd.DataFrame({'coef':LMM1_FM.params,
                       'coef_ci_l':LMM1_FM.conf_int(alpha=0.05).iloc[:, 0],
                       'coef_ci_h':LMM1_FM.conf_int(alpha=0.05).iloc[:, 1],
                       'pval':LMM1_FM.pvalues})
tempDF['coef_ci'] = (tempDF['coef_ci_h'] - tempDF['coef_ci_l'])/2
tempDF = tempDF.reset_index()
sns.set(style='ticks', font='Arial', context='notebook')
plt.figure(figsize=(4, 0.25*len(tempDF)))
plt.errorbar(x=tempDF['coef'], y=tempDF['index'], xerr=tempDF['coef_ci'],
             fmt='ok', ecolor='k', capsize=5)
plt.xlim(-1.0, 1.0)
sns.despine()
plt.xlabel(r'$\beta$'+' or '+r'$\gamma$'+' coefficient in LMM model\n(Mean with 95% CI)')
plt.ylabel('')
plt.axvline(x=0, **{'linestyle':'--', 'color':'k'})
for row_i in range(len(tempDF)):
    if tempDF.iloc[row_i]['pval'] < 0.05:
        plt.axhspan(ymin=row_i-0.5, ymax=row_i+0.5, facecolor='b', alpha=0.2)
plt.gca().invert_yaxis()
plt.margins(y=0.01, tight=True)
plt.show()
print('Elapsed time for fitting LMM:', round(t_elapsed//60), 'min', round(t_elapsed%60, 1), 'sec')

#Baseline BMI-stratified LMM adjusted for general covariates
fe_formula = 'MetBMIchange ~ DaysZ + DaysP2Z + DaysP3Z'\
    ' + C(Sex) + BaseAgeZ + C(Season) + PC1Z + PC2Z + PC3Z + PC4Z + PC5Z'\
    ' + C(BaseBMI_class) + DaysZ:C(BaseBMI_class) + DaysP2Z:C(BaseBMI_class) + DaysP3Z:C(BaseBMI_class)'
model = sm.MixedLM.from_formula(fe_formula, groups='public_client_id',
                                data=tsDF_FM.loc[tsDF_FM['BaseBMI_class']!='Underweight'],
                                re_formula='DaysZ')#Random intercepts for each group are included as default
t_start = time.time()
LMM2_FM = model.fit(method=['bfgs', 'lbfgs', 'cg', 'powell', 'nm'])#Back up for failure of convergence
t_elapsed = time.time() - t_start
print('\n• Model 2: LMM adjusted for baseline BMI-based class and general covariates')
display(LMM2_FM.summary())
print('(Note that the estimates for fixed effects and the variances for random effects are shown in the single table.)')
##Visualize b-coefs and g-coefs
tempDF = pd.DataFrame({'coef':LMM2_FM.params,
                       'coef_ci_l':LMM2_FM.conf_int(alpha=0.05).iloc[:, 0],
                       'coef_ci_h':LMM2_FM.conf_int(alpha=0.05).iloc[:, 1],
                       'pval':LMM2_FM.pvalues})
tempDF['coef_ci'] = (tempDF['coef_ci_h'] - tempDF['coef_ci_l'])/2
tempDF = tempDF.reset_index()
sns.set(style='ticks', font='Arial', context='notebook')
plt.figure(figsize=(4, 0.25*len(tempDF)))
plt.errorbar(x=tempDF['coef'], y=tempDF['index'], xerr=tempDF['coef_ci'],
             fmt='ok', ecolor='k', capsize=5)
plt.xlim(-1.0, 1.0)
sns.despine()
plt.xlabel(r'$\beta$'+' or '+r'$\gamma$'+' coefficient in LMM model\n(Mean with 95% CI)')
plt.ylabel('')
plt.axvline(x=0, **{'linestyle':'--', 'color':'k'})
for row_i in range(len(tempDF)):
    if tempDF.iloc[row_i]['pval'] < 0.05:
        plt.axhspan(ymin=row_i-0.5, ymax=row_i+0.5, facecolor='b', alpha=0.2)
plt.gca().invert_yaxis()
plt.margins(y=0.01, tight=True)
plt.show()
print('Elapsed time for fitting LMM:', round(t_elapsed//60), 'min', round(t_elapsed%60, 1), 'sec')

### 1-4. Estimated transition of biological BMI

> The days in program of the fitted values were different between individuals.  
> –> In statsmodels, MixedLMResults.predict() method use only the fixed effects parameters for prediction.  
> ***–> To evaluate the variability in the in-sample population, the rondom effects component should be added manually.***

In [ ]:
#Sex-stratified model-predicted biological BMI
print('Sex-stratified model-predicted biological BMI')

#Prepare DF to impute days_in_program but maintain in-sample population
tempA1 = tsDF_FM['public_client_id'].unique()
tempA2 = np.arange(0, 365.25/12*month_threshold + 1, 365.25/12*month_threshold/nknots)
tempL = ['Spring', 'Summer', 'Autumn', 'Winter']
tempDF = pd.DataFrame(data={'public_client_id':np.repeat(tempA1, len(tempA2)*len(tempL)),
                            'days_in_program':np.tile(np.repeat(tempA2, len(tempL)), len(tempA1)),
                            'Season':np.tile(tempL, len(tempA1)*len(tempA2))})
tempDF1 = tsDF_FM[['public_client_id', 'BaseMetBMI_class',
                   'BaseBMI_class', 'Sex', 'BaseAge', 'PC1', 'PC2', 'PC3', 'PC4', 'PC5']]
tempDF1 = tempDF1.drop_duplicates('public_client_id', keep='first')
tempDF = pd.merge(tempDF, tempDF1, on='public_client_id', how='left')
##Add dummy variables for the linear regression spline for time
tempL = [365.25/12 * month_threshold/nknots * (knot+1) for knot in range(nknots)]#Days of knots
tempL1 = []#2nd piecewise
tempL2 = []#3rd piecewise
for days in tempDF['days_in_program'].tolist():
    if days < tempL[0]:#[0 month, 6 months)
        tempL1.append(0.0)
        tempL2.append(0.0)
    elif days < tempL[1]:#[6 months, 12 months)
        tempL1.append(days - tempL[0])
        tempL2.append(0.0)
    elif days <= tempL[2]:#[12 months, 18 months]
        tempL1.append(days - tempL[0])
        tempL2.append(days - tempL[1])
tempDF['DaysP2'] = tempL1
tempDF['DaysP3'] = tempL2
##Add a dummy index to use when merging later
tempDF['KeyIndex'] = tempDF['public_client_id'] + '_day' + tempDF['days_in_program'].astype(str)
tempDF = tempDF.set_index('KeyIndex')
##Standardize numeric independent variables using the already fitted StandardScaler
tempL = ['days_in_program', 'DaysP2', 'DaysP3', 'BaseAge', 'PC1', 'PC2', 'PC3', 'PC4', 'PC5']
tempDF1 = pd.DataFrame(scaler_FM.transform(tempDF[tempL]), index=tempDF.index,
                       columns=['DaysZ', 'DaysP2Z', 'DaysP3Z', 'BaseAgeZ', 'PC1Z', 'PC2Z', 'PC3Z', 'PC4Z', 'PC5Z'])
tempDF = pd.merge(tempDF, tempDF1, left_index=True, right_index=True, how='inner')

#Model 1 prediction
##Predicted component from the fixed effects
tempS = LMM1_FM.predict(tempDF)
##Create design matrix for the random effects
tempDF1 = pd.DataFrame({'Intercept_re':np.repeat(1, len(tempDF)),
                        'Slope_re':tempDF['DaysZ']})
tempD = LMM1_FM.random_effects#Dictionary of key=group label and value=pd.Series
##Predicted component from the random effects
tempL = [np.dot(tempDF1.iloc[row_i], tempD[group_n])
         for (row_i, group_n) in enumerate(tempDF['public_client_id'].tolist())]
##Overall prediction for in-sample individuals
tempDF['MetBMIchange_predict1'] = tempS + tempL

#Model 2 prediction
##Eliminate 'Underweight' class
tempDF1 = tempDF.loc[tempDF['BaseBMI_class']!='Underweight']
##Predicted component from the fixed effects
tempS = LMM2_FM.predict(tempDF1)
##Create design matrix for the random effects
tempDF2 = pd.DataFrame({'Intercept_re':np.repeat(1, len(tempDF1)),
                        'Slope_re':tempDF1['DaysZ']})
tempD = LMM2_FM.random_effects#Dictionary of key=group label and value=pd.Series
##Predicted component from the random effects
tempL = [np.dot(tempDF2.iloc[row_i], tempD[group_n])
         for (row_i, group_n) in enumerate(tempDF1['public_client_id'].tolist())]
##Overall prediction for in-sample individuals
tempDF1['MetBMIchange_predict2'] = tempS + tempL
##Add prediction to the whole DF
tempDF = pd.merge(tempDF, tempDF1['MetBMIchange_predict2'],
                  left_index=True, right_index=True, how='outer')

#Flatten effects of the dummy season variable
tempDF = tempDF.groupby(by=['public_client_id', 'days_in_program'])
tempDF = tempDF.agg({'MetBMIchange_predict1':'mean', 'MetBMIchange_predict2':'mean'})
tempDF = tempDF.reset_index()
##Recover the dropped covariates and save the prediction DF to use later
tempDF1 = tsDF_FM[['public_client_id', 'BaseMetBMI_class', 'BaseBMI_class', 'Sex']]
tempDF1 = tempDF1.drop_duplicates('public_client_id', keep='first')
tempDF = pd.merge(tempDF, tempDF1, on='public_client_id', how='left')
tempDF['KeyIndex'] = tempDF['public_client_id'] + '_day' + tempDF['days_in_program'].astype(str)
metLMM_FM = tempDF.set_index('KeyIndex')
print('\n• Prediction DF')
display(metLMM_FM)
display(metLMM_FM.describe(include='all'))

#Plot1
print('\n• Model 1: LMM adjusted for general covariates')
sns.set(style='ticks', font='Arial', context='notebook')
plt.figure(figsize=(5, 3))
sns.lineplot(data=tsDF_FM, x='days_in_program', y='MetBMIchange',
             units='public_client_id', estimator=None, color='gray', lw=1, alpha=0.1, legend=None)
sns.scatterplot(data=tsDF_FM, x='days_in_program', y='MetBMIchange',
                color='gray', edgecolor='k', s=20, alpha=0.3)
p = sns.lineplot(data=metLMM_FM, x='days_in_program', y='MetBMIchange_predict1',
                 estimator='mean', ci=95, color='b')
p.set(xlim=(0, 365.25/12*month_threshold), xticks=np.arange(0, 365.25/12*month_threshold, 100),
      ylim=(-20, 15), yticks=np.arange(-20, 15.1, 5))
sns.despine()
plt.xlabel('Days in program')
plt.ylabel('Rate of change [%]')
plt.show()

#Plot2
print('\n• Model 2: LMM adjusted for baseline BMI-based class and general covariates')
tempD = {'Normal':'green', 'Overweight':'orange', 'Obese':'red'}
sns.set(style='ticks', font='Arial', context='notebook')
plt.figure(figsize=(5, 3))
sns.scatterplot(data=tsDF_FM[tsDF_FM['BaseBMI_class']!='Underweight'], x='days_in_program', y='MetBMIchange',
                hue='BaseBMI_class', hue_order=tempD.keys(), palette=tempD,
                edgecolor='k', s=20, alpha=0.3)
p = sns.lineplot(data=metLMM_FM, x='days_in_program', y='MetBMIchange_predict2',
                 hue='BaseBMI_class', hue_order=tempD.keys(), palette=tempD,
                 estimator='mean', ci=95)
p.set(xlim=(0, 365.25/12*month_threshold), xticks=np.arange(0, 365.25/12*month_threshold, 100),
      ylim=(-20, 15), yticks=np.arange(-20, 15.1, 5))
sns.despine()
plt.xlabel('Days in program')
plt.ylabel('Rate of change [%]')
plt.legend(bbox_to_anchor=(1, 0.5), loc='center left', borderaxespad=1)
plt.show()

## 2. Proteomics

### 2-1. Prepare the time-series DF

In [ ]:
#Import time-series biological BMI
print('Sex-stratified model')
fileDir = './ExportData/'
fileName = '210106_Biological-BMI-paper_longitudinal-LASSO_times-series-protBMI-FemaleMale.csv'
tsDF_FM = pd.read_csv(fileDir+fileName, dtype={'public_client_id': str}).set_index('KeyIndex')
display(tsDF_FM)

### 2-2. Prepare DFs for LMM with the linear regresion spline for time

In [ ]:
#Check measurement distribution
sns.set(style='ticks', font='Arial', context='notebook')
plt.figure(figsize=(4, 3))
sns.distplot(tsDF_FM['days_in_program'], color='r')
sns.despine()
plt.axvline(x=365.25/12*0, **{'linestyle':'--', 'color':'gray'})
plt.axvline(x=365.25/12*6, **{'linestyle':'--', 'color':'gray'})
plt.axvline(x=365.25/12*12, **{'linestyle':'--', 'color':'gray'})
plt.axvline(x=365.25/12*18, **{'linestyle':'--', 'color':'gray'})
plt.axvspan(xmin=365.25/12*18, xmax=plt.gca().get_xlim()[1], facecolor='gray', alpha=0.2)
plt.ylabel('Density')
plt.xlabel('Days in program')
plt.show()

In [ ]:
#Sex-stratified model
print('Sex-stratified model')

#Drop unnecessary variables
tempDF = tsDF_FM[['public_client_id', 'days_in_program', 'Season', 'ProtBMI', 'BaseProtBMI',
                  'BaseProtBMI_class', 'BaseBMI_class', 'Sex', 'BaseAge', 'PC1', 'PC2', 'PC3', 'PC4', 'PC5']]
print('Time-series DF nrows before elimination:', len(tempDF))
print(' -> Unique ID:', len(tempDF['public_client_id'].unique()))

#Select measurements used in the longitudinal analysis
month_threshold = 18
tempDF = tempDF.loc[tempDF['days_in_program'] <= 365.25/12*month_threshold]
print('Time-series DF nrows after selecting measurements during the taget period:', len(tempDF))
print(' -> Unique ID:', len(tempDF['public_client_id'].unique()))

#Select the cohort used in the longitudinal analysis
##Select participants who have more than 2 measuremnts in BMI
fileDir = './ExportData/'
fileName = '210104_Biological-BMI-paper_data-cleaning-BMI-omics_time-series-bmiDF-without-imputation_final-cohort.tsv'
tempDF1 = pd.read_csv(fileDir+fileName, sep='\t', dtype={'public_client_id': str}).set_index('public_client_id')
tempDF1 = tempDF1.loc[tempDF1['days_in_program'] <= 365.25/12*month_threshold]
tempS = tempDF1.index.value_counts()
tempL1 = tempS.loc[tempS>=2].index.tolist()
##Select participants who have more than 2 measuremnts in omics
fileDir = './ExportData/'
fileName = '210104_Biological-BMI-paper_data-cleaning-BMI-omics_time-series-combiDF-without-imputation_final-cohort.tsv'
tempDF1 = pd.read_csv(fileDir+fileName, sep='\t', dtype={'public_client_id': str}).set_index('public_client_id')
tempDF1 = tempDF1.loc[tempDF1['days_in_program'] <= 365.25/12*month_threshold]
tempS = tempDF1.index.value_counts()
tempL2 = tempS.loc[tempS>=2].index.tolist()
##Select participants who have more than 2 measuremnts in both BMI and omics
tempL = list(set(tempL1) & set(tempL2))
print('Participants who have more than 2 measurements in both BMI and omics:', len(tempL))
tempDF = tempDF.loc[tempDF['public_client_id'].isin(tempL)]
print('Time-series DF nrows after selecting participants who have ≥2 measuremnts in both BMI and omics:',
      len(tempDF))
print(' -> Unique ID:', len(tempDF['public_client_id'].unique()))

#Select the participants who have all covariates
tempDF = tempDF.dropna()
print('Time-series DF nrows after selecting participants who have all covariates:', len(tempDF))
print(' -> Unique ID:', len(tempDF['public_client_id'].unique()))

#Check measurement distribution
sns.set(style='ticks', font='Arial', context='talk')
plt.figure(figsize=(4, 3))
sns.distplot(tempDF['days_in_program'], color='r')
sns.despine()
plt.axvline(x=365.25/12*0, **{'linestyle':'--', 'color':'gray'})
plt.axvline(x=365.25/12*6, **{'linestyle':'--', 'color':'gray'})
plt.axvline(x=365.25/12*12, **{'linestyle':'--', 'color':'gray'})
plt.axvline(x=365.25/12*18, **{'linestyle':'--', 'color':'gray'})
plt.axvspan(xmin=365.25/12*18, xmax=plt.gca().get_xlim()[1], facecolor='gray', alpha=0.2)
plt.ylabel('Density')
plt.xlabel('Days in program')
plt.show()

#Calculate rate of change
tempDF['ProtBMIchange'] = (tempDF['ProtBMI'] - tempDF['BaseProtBMI']) / tempDF['BaseProtBMI'] * 100

#Check distribution
print('Skewness of rate of change:', stats.skew(tempDF['ProtBMIchange']))
sns.set(style='ticks', font='Arial', context='notebook')
plt.figure(figsize=(4, 3))
sns.distplot(tempDF['ProtBMIchange'], color='r')
sns.despine()
plt.ylabel('Density')
plt.xlabel('Rate of change [%]')
plt.show()

#Add dummy variables for the linear regression spline for time
nknots = 3
tempL = [365.25/12 * month_threshold/nknots * (knot+1) for knot in range(nknots)]#Days of knots
tempL1 = []#2nd piece
tempL2 = []#3rd piece
for days in tempDF['days_in_program'].tolist():
    if days < tempL[0]:#[0 month, 6 months)
        tempL1.append(0.0)
        tempL2.append(0.0)
    elif days < tempL[1]:#[6 months, 12 months)
        tempL1.append(days - tempL[0])
        tempL2.append(0.0)
    elif days <= tempL[2]:#[12 months, 18 months]
        tempL1.append(days - tempL[0])
        tempL2.append(days - tempL[1])
tempDF['DaysP2'] = tempL1
tempDF['DaysP3'] = tempL2

#Sort to make 'Normal' the standard in the baseline BMI-stratified LMM (Normal < Obese < Overweight)
tempDF = tempDF.sort_values(by=['BaseBMI_class'], ascending=True)

#Standardize numeric independent variables
scaler_FM = StandardScaler(copy=True, with_mean=True, with_std=True)#Use later again during predictions
tempL = ['days_in_program', 'DaysP2', 'DaysP3', 'BaseAge', 'PC1', 'PC2', 'PC3', 'PC4', 'PC5']
scaler_FM.fit(tempDF[tempL])
tempDF1 = pd.DataFrame(scaler_FM.transform(tempDF[tempL]), index=tempDF.index,
                       columns=['DaysZ', 'DaysP2Z', 'DaysP3Z', 'BaseAgeZ', 'PC1Z', 'PC2Z', 'PC3Z', 'PC4Z', 'PC5Z'])
tempDF = pd.merge(tempDF, tempDF1, left_index=True, right_index=True, how='inner')

#Confirm examples
sns.set(style='ticks', font='Arial', context='notebook')
plt.figure(figsize=(4, 3))
sns.distplot(tempDF['DaysZ'], label='DaysZ')
sns.distplot(tempDF['BaseAgeZ'], label='BaseAgeZ')
sns.distplot(tempDF['PC1Z'], label='PC1Z')
sns.despine()
plt.ylabel('Density')
plt.xlabel('Z-score')
tempL = [365.25/12 * month_threshold/nknots * (knot+1) for knot in range(nknots)]#Days of knots
tempS = (pd.Series(tempL) - scaler_FM.mean_[0]) / scaler_FM.scale_[0]#Z-score transformation
for knot in tempS:
    plt.axvline(x=knot, **{'linestyle':'--', 'color':'gray'})
plt.legend(bbox_to_anchor=(1, 0.5), loc='center left', borderaxespad=1)
plt.show()

#Update
tsDF_FM = tempDF
display(tsDF_FM)

### 2-3. LMM with random intercepts and random slopes for time

> Days in the program is fitted as a numeric variable using linear regression splines (cf. Zubair, N. et al. Sci. Rep. 2019).  
> ***–> Allows differences in the trajectory of biological BMI change.***
>
> Random intercepts and random slopes for days in the program are used as random effects (cf. Zubair, N. et al. Sci. Rep. 2019).  
> ***–> Allows individual differences in the intercept and the rate of change of biological BMI over the program.***  
> –> Random slope is applied not for the piecewise variable of time but only for the overall variable of time; otherwise, leads to "LinAlgError: Singular matrix" due to small sample size per most individuals. Also, it is not so weird to assume the random slope is consistent during the program.  
>
> ***Eliminate "Underweight" class in the baseline obesity class-stratified models.***  
> <– Otherwise, leads to "LinAlgError: Singular matrix" due to small sample population.  

In [ ]:
#Sex-stratified model-predicted biological BMI
print('Sex-stratified model-predicted biological BMI')

#Overall LMM adjusted for general covariates
fe_formula = 'ProtBMIchange ~ DaysZ + DaysP2Z + DaysP3Z'\
    ' + C(Sex) + BaseAgeZ + C(Season) + PC1Z + PC2Z + PC3Z + PC4Z + PC5Z'
model = sm.MixedLM.from_formula(fe_formula, groups='public_client_id', data=tsDF_FM,
                                re_formula='DaysZ')#Random intercepts for each group are included as default
t_start = time.time()
LMM1_FM = model.fit(method=['bfgs', 'lbfgs', 'cg', 'powell', 'nm'])#Back up for failure of convergence
t_elapsed = time.time() - t_start
print('\n• Model 1: LMM adjusted for general covariates')
display(LMM1_FM.summary())
print('(Note that the estimates for fixed effects and the variances for random effects are shown in the single table.)')
##Visualize b-coefs and g-coefs
tempDF = pd.DataFrame({'coef':LMM1_FM.params,
                       'coef_ci_l':LMM1_FM.conf_int(alpha=0.05).iloc[:, 0],
                       'coef_ci_h':LMM1_FM.conf_int(alpha=0.05).iloc[:, 1],
                       'pval':LMM1_FM.pvalues})
tempDF['coef_ci'] = (tempDF['coef_ci_h'] - tempDF['coef_ci_l'])/2
tempDF = tempDF.reset_index()
sns.set(style='ticks', font='Arial', context='notebook')
plt.figure(figsize=(4, 0.25*len(tempDF)))
plt.errorbar(x=tempDF['coef'], y=tempDF['index'], xerr=tempDF['coef_ci'],
             fmt='ok', ecolor='k', capsize=5)
plt.xlim(-1.0, 1.0)
sns.despine()
plt.xlabel(r'$\beta$'+' or '+r'$\gamma$'+' coefficient in LMM model\n(Mean with 95% CI)')
plt.ylabel('')
plt.axvline(x=0, **{'linestyle':'--', 'color':'k'})
for row_i in range(len(tempDF)):
    if tempDF.iloc[row_i]['pval'] < 0.05:
        plt.axhspan(ymin=row_i-0.5, ymax=row_i+0.5, facecolor='r', alpha=0.2)
plt.gca().invert_yaxis()
plt.margins(y=0.01, tight=True)
plt.show()
print('Elapsed time for fitting LMM:', round(t_elapsed//60), 'min', round(t_elapsed%60, 1), 'sec')

#Baseline BMI-stratified LMM adjusted for general covariates
fe_formula = 'ProtBMIchange ~ DaysZ + DaysP2Z + DaysP3Z'\
    ' + C(Sex) + BaseAgeZ + C(Season) + PC1Z + PC2Z + PC3Z + PC4Z + PC5Z'\
    ' + C(BaseBMI_class) + DaysZ:C(BaseBMI_class) + DaysP2Z:C(BaseBMI_class) + DaysP3Z:C(BaseBMI_class)'
model = sm.MixedLM.from_formula(fe_formula, groups='public_client_id',
                                data=tsDF_FM.loc[tsDF_FM['BaseBMI_class']!='Underweight'],
                                re_formula='DaysZ')#Random intercepts for each group are included as default
t_start = time.time()
LMM2_FM = model.fit(method=['bfgs', 'lbfgs', 'cg', 'powell', 'nm'])#Back up for failure of convergence
t_elapsed = time.time() - t_start
print('\n• Model 2: LMM adjusted for baseline BMI-based class and general covariates')
display(LMM2_FM.summary())
print('(Note that the estimates for fixed effects and the variances for random effects are shown in the single table.)')
##Visualize b-coefs and g-coefs
tempDF = pd.DataFrame({'coef':LMM2_FM.params,
                       'coef_ci_l':LMM2_FM.conf_int(alpha=0.05).iloc[:, 0],
                       'coef_ci_h':LMM2_FM.conf_int(alpha=0.05).iloc[:, 1],
                       'pval':LMM2_FM.pvalues})
tempDF['coef_ci'] = (tempDF['coef_ci_h'] - tempDF['coef_ci_l'])/2
tempDF = tempDF.reset_index()
sns.set(style='ticks', font='Arial', context='notebook')
plt.figure(figsize=(4, 0.25*len(tempDF)))
plt.errorbar(x=tempDF['coef'], y=tempDF['index'], xerr=tempDF['coef_ci'],
             fmt='ok', ecolor='k', capsize=5)
plt.xlim(-1.0, 1.0)
sns.despine()
plt.xlabel(r'$\beta$'+' or '+r'$\gamma$'+' coefficient in LMM model\n(Mean with 95% CI)')
plt.ylabel('')
plt.axvline(x=0, **{'linestyle':'--', 'color':'k'})
for row_i in range(len(tempDF)):
    if tempDF.iloc[row_i]['pval'] < 0.05:
        plt.axhspan(ymin=row_i-0.5, ymax=row_i+0.5, facecolor='r', alpha=0.2)
plt.gca().invert_yaxis()
plt.margins(y=0.01, tight=True)
plt.show()
print('Elapsed time for fitting LMM:', round(t_elapsed//60), 'min', round(t_elapsed%60, 1), 'sec')

### 2-4. Estimated transition of biological BMI

> The days in program of the fitted values were different between individuals.  
> –> In statsmodels, MixedLMResults.predict() method use only the fixed effects parameters for prediction.  
> ***–> To evaluate the variability in the in-sample population, the rondom effects component should be added manually.***

In [ ]:
#Sex-stratified model-predicted biological BMI
print('Sex-stratified model-predicted biological BMI')

#Prepare DF to impute days_in_program but maintain in-sample population
tempA1 = tsDF_FM['public_client_id'].unique()
tempA2 = np.arange(0, 365.25/12*month_threshold + 1, 365.25/12*month_threshold/nknots)
tempL = ['Spring', 'Summer', 'Autumn', 'Winter']
tempDF = pd.DataFrame(data={'public_client_id':np.repeat(tempA1, len(tempA2)*len(tempL)),
                            'days_in_program':np.tile(np.repeat(tempA2, len(tempL)), len(tempA1)),
                            'Season':np.tile(tempL, len(tempA1)*len(tempA2))})
tempDF1 = tsDF_FM[['public_client_id', 'BaseProtBMI_class',
                   'BaseBMI_class', 'Sex', 'BaseAge', 'PC1', 'PC2', 'PC3', 'PC4', 'PC5']]
tempDF1 = tempDF1.drop_duplicates('public_client_id', keep='first')
tempDF = pd.merge(tempDF, tempDF1, on='public_client_id', how='left')
##Add dummy variables for the linear regression spline for time
tempL = [365.25/12 * month_threshold/nknots * (knot+1) for knot in range(nknots)]#Days of knots
tempL1 = []#2nd piecewise
tempL2 = []#3rd piecewise
for days in tempDF['days_in_program'].tolist():
    if days < tempL[0]:#[0 month, 6 months)
        tempL1.append(0.0)
        tempL2.append(0.0)
    elif days < tempL[1]:#[6 months, 12 months)
        tempL1.append(days - tempL[0])
        tempL2.append(0.0)
    elif days <= tempL[2]:#[12 months, 18 months]
        tempL1.append(days - tempL[0])
        tempL2.append(days - tempL[1])
tempDF['DaysP2'] = tempL1
tempDF['DaysP3'] = tempL2
##Add a dummy index to use when merging later
tempDF['KeyIndex'] = tempDF['public_client_id'] + '_day' + tempDF['days_in_program'].astype(str)
tempDF = tempDF.set_index('KeyIndex')
##Standardize numeric independent variables using the already fitted StandardScaler
tempL = ['days_in_program', 'DaysP2', 'DaysP3', 'BaseAge', 'PC1', 'PC2', 'PC3', 'PC4', 'PC5']
tempDF1 = pd.DataFrame(scaler_FM.transform(tempDF[tempL]), index=tempDF.index,
                       columns=['DaysZ', 'DaysP2Z', 'DaysP3Z', 'BaseAgeZ', 'PC1Z', 'PC2Z', 'PC3Z', 'PC4Z', 'PC5Z'])
tempDF = pd.merge(tempDF, tempDF1, left_index=True, right_index=True, how='inner')

#Model 1 prediction
##Predicted component from the fixed effects
tempS = LMM1_FM.predict(tempDF)
##Create design matrix for the random effects
tempDF1 = pd.DataFrame({'Intercept_re':np.repeat(1, len(tempDF)),
                        'Slope_re':tempDF['DaysZ']})
tempD = LMM1_FM.random_effects#Dictionary of key=group label and value=pd.Series
##Predicted component from the random effects
tempL = [np.dot(tempDF1.iloc[row_i], tempD[group_n])
         for (row_i, group_n) in enumerate(tempDF['public_client_id'].tolist())]
##Overall prediction for in-sample individuals
tempDF['ProtBMIchange_predict1'] = tempS + tempL

#Model 2 prediction
##Eliminate 'Underweight' class
tempDF1 = tempDF.loc[tempDF['BaseBMI_class']!='Underweight']
##Predicted component from the fixed effects
tempS = LMM2_FM.predict(tempDF1)
##Create design matrix for the random effects
tempDF2 = pd.DataFrame({'Intercept_re':np.repeat(1, len(tempDF1)),
                        'Slope_re':tempDF1['DaysZ']})
tempD = LMM2_FM.random_effects#Dictionary of key=group label and value=pd.Series
##Predicted component from the random effects
tempL = [np.dot(tempDF2.iloc[row_i], tempD[group_n])
         for (row_i, group_n) in enumerate(tempDF1['public_client_id'].tolist())]
##Overall prediction for in-sample individuals
tempDF1['ProtBMIchange_predict2'] = tempS + tempL
##Add prediction to the whole DF
tempDF = pd.merge(tempDF, tempDF1['ProtBMIchange_predict2'],
                  left_index=True, right_index=True, how='outer')

#Flatten effects of the dummy season variable
tempDF = tempDF.groupby(by=['public_client_id', 'days_in_program'])
tempDF = tempDF.agg({'ProtBMIchange_predict1':'mean', 'ProtBMIchange_predict2':'mean'})
tempDF = tempDF.reset_index()
##Recover the dropped covariates and save the prediction DF to use later
tempDF1 = tsDF_FM[['public_client_id', 'BaseProtBMI_class', 'BaseBMI_class', 'Sex']]
tempDF1 = tempDF1.drop_duplicates('public_client_id', keep='first')
tempDF = pd.merge(tempDF, tempDF1, on='public_client_id', how='left')
tempDF['KeyIndex'] = tempDF['public_client_id'] + '_day' + tempDF['days_in_program'].astype(str)
protLMM_FM = tempDF.set_index('KeyIndex')
print('\n• Prediction DF')
display(protLMM_FM)
display(protLMM_FM.describe(include='all'))

#Plot1
print('\n• Model 1: LMM adjusted for general covariates')
sns.set(style='ticks', font='Arial', context='notebook')
plt.figure(figsize=(5, 3))
sns.lineplot(data=tsDF_FM, x='days_in_program', y='ProtBMIchange',
             units='public_client_id', estimator=None, color='gray', lw=1, alpha=0.1, legend=None)
sns.scatterplot(data=tsDF_FM, x='days_in_program', y='ProtBMIchange',
                color='gray', edgecolor='k', s=20, alpha=0.3)
p = sns.lineplot(data=protLMM_FM, x='days_in_program', y='ProtBMIchange_predict1',
                 estimator='mean', ci=95, color='r')
p.set(xlim=(0, 365.25/12*month_threshold), xticks=np.arange(0, 365.25/12*month_threshold, 100),
      ylim=(-20, 15), yticks=np.arange(-20, 15.1, 5))
sns.despine()
plt.xlabel('Days in program')
plt.ylabel('Rate of change [%]')
plt.show()

#Plot2
print('\n• Model 2: LMM adjusted for baseline BMI-based class and general covariates')
tempD = {'Normal':'green', 'Overweight':'orange', 'Obese':'red'}
sns.set(style='ticks', font='Arial', context='notebook')
plt.figure(figsize=(5, 3))
sns.scatterplot(data=tsDF_FM[tsDF_FM['BaseBMI_class']!='Underweight'], x='days_in_program', y='ProtBMIchange',
                hue='BaseBMI_class', hue_order=tempD.keys(), palette=tempD,
                edgecolor='k', s=20, alpha=0.3)
p = sns.lineplot(data=protLMM_FM, x='days_in_program', y='ProtBMIchange_predict2',
                 hue='BaseBMI_class', hue_order=tempD.keys(), palette=tempD,
                 estimator='mean', ci=95)
p.set(xlim=(0, 365.25/12*month_threshold), xticks=np.arange(0, 365.25/12*month_threshold, 100),
      ylim=(-20, 15), yticks=np.arange(-20, 15.1, 5))
sns.despine()
plt.xlabel('Days in program')
plt.ylabel('Rate of change [%]')
plt.legend(bbox_to_anchor=(1, 0.5), loc='center left', borderaxespad=1)
plt.show()

## 3. Clinical labs

### 3-1. Prepare the time-series DF

In [ ]:
#Import time-series biological BMI
print('Sex-stratified model')
fileDir = './ExportData/'
fileName = '210106_Biological-BMI-paper_longitudinal-LASSO_times-series-chemBMI-FemaleMale.csv'
tsDF_FM = pd.read_csv(fileDir+fileName, dtype={'public_client_id': str}).set_index('KeyIndex')
display(tsDF_FM)

### 3-2. Prepare DFs for LMM with the linear regresion spline for time

In [ ]:
#Check measurement distribution
sns.set(style='ticks', font='Arial', context='notebook')
plt.figure(figsize=(4, 3))
sns.distplot(tsDF_FM['days_in_program'], color='g')
sns.despine()
plt.axvline(x=365.25/12*0, **{'linestyle':'--', 'color':'gray'})
plt.axvline(x=365.25/12*6, **{'linestyle':'--', 'color':'gray'})
plt.axvline(x=365.25/12*12, **{'linestyle':'--', 'color':'gray'})
plt.axvline(x=365.25/12*18, **{'linestyle':'--', 'color':'gray'})
plt.axvspan(xmin=365.25/12*18, xmax=plt.gca().get_xlim()[1], facecolor='gray', alpha=0.2)
plt.ylabel('Density')
plt.xlabel('Days in program')
plt.show()

In [ ]:
#Sex-stratified model
print('Sex-stratified model')

#Drop unnecessary variables
tempDF = tsDF_FM[['public_client_id', 'days_in_program', 'Season', 'ChemBMI', 'BaseChemBMI',
                  'BaseChemBMI_class', 'BaseBMI_class', 'Sex', 'BaseAge', 'PC1', 'PC2', 'PC3', 'PC4', 'PC5']]
print('Time-series DF nrows before elimination:', len(tempDF))
print(' -> Unique ID:', len(tempDF['public_client_id'].unique()))

#Select measurements used in the longitudinal analysis
month_threshold = 18
tempDF = tempDF.loc[tempDF['days_in_program'] <= 365.25/12*month_threshold]
print('Time-series DF nrows after selecting measurements during the taget period:', len(tempDF))
print(' -> Unique ID:', len(tempDF['public_client_id'].unique()))

#Select the cohort used in the longitudinal analysis
##Select participants who have more than 2 measuremnts in BMI
fileDir = './ExportData/'
fileName = '210104_Biological-BMI-paper_data-cleaning-BMI-omics_time-series-bmiDF-without-imputation_final-cohort.tsv'
tempDF1 = pd.read_csv(fileDir+fileName, sep='\t', dtype={'public_client_id': str}).set_index('public_client_id')
tempDF1 = tempDF1.loc[tempDF1['days_in_program'] <= 365.25/12*month_threshold]
tempS = tempDF1.index.value_counts()
tempL1 = tempS.loc[tempS>=2].index.tolist()
##Select participants who have more than 2 measuremnts in omics
fileDir = './ExportData/'
fileName = '210104_Biological-BMI-paper_data-cleaning-BMI-omics_time-series-combiDF-without-imputation_final-cohort.tsv'
tempDF1 = pd.read_csv(fileDir+fileName, sep='\t', dtype={'public_client_id': str}).set_index('public_client_id')
tempDF1 = tempDF1.loc[tempDF1['days_in_program'] <= 365.25/12*month_threshold]
tempS = tempDF1.index.value_counts()
tempL2 = tempS.loc[tempS>=2].index.tolist()
##Select participants who have more than 2 measuremnts in both BMI and omics
tempL = list(set(tempL1) & set(tempL2))
print('Participants who have more than 2 measurements in both BMI and omics:', len(tempL))
tempDF = tempDF.loc[tempDF['public_client_id'].isin(tempL)]
print('Time-series DF nrows after selecting participants who have ≥2 measuremnts in both BMI and omics:',
      len(tempDF))
print(' -> Unique ID:', len(tempDF['public_client_id'].unique()))

#Select the participants who have all covariates
tempDF = tempDF.dropna()
print('Time-series DF nrows after selecting participants who have all covariates:', len(tempDF))
print(' -> Unique ID:', len(tempDF['public_client_id'].unique()))

#Check measurement distribution
sns.set(style='ticks', font='Arial', context='talk')
plt.figure(figsize=(4, 3))
sns.distplot(tempDF['days_in_program'], color='g')
sns.despine()
plt.axvline(x=365.25/12*0, **{'linestyle':'--', 'color':'gray'})
plt.axvline(x=365.25/12*6, **{'linestyle':'--', 'color':'gray'})
plt.axvline(x=365.25/12*12, **{'linestyle':'--', 'color':'gray'})
plt.axvline(x=365.25/12*18, **{'linestyle':'--', 'color':'gray'})
plt.axvspan(xmin=365.25/12*18, xmax=plt.gca().get_xlim()[1], facecolor='gray', alpha=0.2)
plt.ylabel('Density')
plt.xlabel('Days in program')
plt.show()

#Calculate rate of change
tempDF['ChemBMIchange'] = (tempDF['ChemBMI'] - tempDF['BaseChemBMI']) / tempDF['BaseChemBMI'] * 100

#Check distribution
print('Skewness of rate of change:', stats.skew(tempDF['ChemBMIchange']))
sns.set(style='ticks', font='Arial', context='notebook')
plt.figure(figsize=(4, 3))
sns.distplot(tempDF['ChemBMIchange'], color='g')
sns.despine()
plt.ylabel('Density')
plt.xlabel('Rate of change [%]')
plt.show()

#Add dummy variables for the linear regression spline for time
nknots = 3
tempL = [365.25/12 * month_threshold/nknots * (knot+1) for knot in range(nknots)]#Days of knots
tempL1 = []#2nd piece
tempL2 = []#3rd piece
for days in tempDF['days_in_program'].tolist():
    if days < tempL[0]:#[0 month, 6 months)
        tempL1.append(0.0)
        tempL2.append(0.0)
    elif days < tempL[1]:#[6 months, 12 months)
        tempL1.append(days - tempL[0])
        tempL2.append(0.0)
    elif days <= tempL[2]:#[12 months, 18 months]
        tempL1.append(days - tempL[0])
        tempL2.append(days - tempL[1])
tempDF['DaysP2'] = tempL1
tempDF['DaysP3'] = tempL2

#Sort to make 'Normal' the standard in the baseline BMI-stratified LMM (Normal < Obese < Overweight)
tempDF = tempDF.sort_values(by=['BaseBMI_class'], ascending=True)

#Standardize numeric independent variables
scaler_FM = StandardScaler(copy=True, with_mean=True, with_std=True)#Use later again during predictions
tempL = ['days_in_program', 'DaysP2', 'DaysP3', 'BaseAge', 'PC1', 'PC2', 'PC3', 'PC4', 'PC5']
scaler_FM.fit(tempDF[tempL])
tempDF1 = pd.DataFrame(scaler_FM.transform(tempDF[tempL]), index=tempDF.index,
                       columns=['DaysZ', 'DaysP2Z', 'DaysP3Z', 'BaseAgeZ', 'PC1Z', 'PC2Z', 'PC3Z', 'PC4Z', 'PC5Z'])
tempDF = pd.merge(tempDF, tempDF1, left_index=True, right_index=True, how='inner')

#Confirm examples
sns.set(style='ticks', font='Arial', context='notebook')
plt.figure(figsize=(4, 3))
sns.distplot(tempDF['DaysZ'], label='DaysZ')
sns.distplot(tempDF['BaseAgeZ'], label='BaseAgeZ')
sns.distplot(tempDF['PC1Z'], label='PC1Z')
sns.despine()
plt.ylabel('Density')
plt.xlabel('Z-score')
tempL = [365.25/12 * month_threshold/nknots * (knot+1) for knot in range(nknots)]#Days of knots
tempS = (pd.Series(tempL) - scaler_FM.mean_[0]) / scaler_FM.scale_[0]#Z-score transformation
for knot in tempS:
    plt.axvline(x=knot, **{'linestyle':'--', 'color':'gray'})
plt.legend(bbox_to_anchor=(1, 0.5), loc='center left', borderaxespad=1)
plt.show()

#Update
tsDF_FM = tempDF
display(tsDF_FM)

### 3-3. LMM with random intercepts and random slopes for time

> Days in the program is fitted as a numeric variable using linear regression splines (cf. Zubair, N. et al. Sci. Rep. 2019).  
> ***–> Allows differences in the trajectory of biological BMI change.***
>
> Random intercepts and random slopes for days in the program are used as random effects (cf. Zubair, N. et al. Sci. Rep. 2019).  
> ***–> Allows individual differences in the intercept and the rate of change of biological BMI over the program.***  
> –> Random slope is applied not for the piecewise variable of time but only for the overall variable of time; otherwise, leads to "LinAlgError: Singular matrix" due to small sample size per most individuals. Also, it is not so weird to assume the random slope is consistent during the program.  
>
> ***Eliminate "Underweight" class in the baseline obesity class-stratified models.***  
> <– Otherwise, leads to "LinAlgError: Singular matrix" due to small sample population.  

In [ ]:
#Sex-stratified model-predicted biological BMI
print('Sex-stratified model-predicted biological BMI')

#Overall LMM adjusted for general covariates
fe_formula = 'ChemBMIchange ~ DaysZ + DaysP2Z + DaysP3Z'\
    ' + C(Sex) + BaseAgeZ + C(Season) + PC1Z + PC2Z + PC3Z + PC4Z + PC5Z'
model = sm.MixedLM.from_formula(fe_formula, groups='public_client_id', data=tsDF_FM,
                                re_formula='DaysZ')#Random intercepts for each group are included as default
t_start = time.time()
LMM1_FM = model.fit(method=['bfgs', 'lbfgs', 'cg', 'powell', 'nm'])#Back up for failure of convergence
t_elapsed = time.time() - t_start
print('\n• Model 1: LMM adjusted for general covariates')
display(LMM1_FM.summary())
print('(Note that the estimates for fixed effects and the variances for random effects are shown in the single table.)')
##Visualize b-coefs and g-coefs
tempDF = pd.DataFrame({'coef':LMM1_FM.params,
                       'coef_ci_l':LMM1_FM.conf_int(alpha=0.05).iloc[:, 0],
                       'coef_ci_h':LMM1_FM.conf_int(alpha=0.05).iloc[:, 1],
                       'pval':LMM1_FM.pvalues})
tempDF['coef_ci'] = (tempDF['coef_ci_h'] - tempDF['coef_ci_l'])/2
tempDF = tempDF.reset_index()
sns.set(style='ticks', font='Arial', context='notebook')
plt.figure(figsize=(4, 0.25*len(tempDF)))
plt.errorbar(x=tempDF['coef'], y=tempDF['index'], xerr=tempDF['coef_ci'],
             fmt='ok', ecolor='k', capsize=5)
plt.xlim(-1.0, 1.0)
sns.despine()
plt.xlabel(r'$\beta$'+' or '+r'$\gamma$'+' coefficient in LMM model\n(Mean with 95% CI)')
plt.ylabel('')
plt.axvline(x=0, **{'linestyle':'--', 'color':'k'})
for row_i in range(len(tempDF)):
    if tempDF.iloc[row_i]['pval'] < 0.05:
        plt.axhspan(ymin=row_i-0.5, ymax=row_i+0.5, facecolor='g', alpha=0.2)
plt.gca().invert_yaxis()
plt.margins(y=0.01, tight=True)
plt.show()
print('Elapsed time for fitting LMM:', round(t_elapsed//60), 'min', round(t_elapsed%60, 1), 'sec')

#Baseline BMI-stratified LMM adjusted for general covariates
fe_formula = 'ChemBMIchange ~ DaysZ + DaysP2Z + DaysP3Z'\
    ' + C(Sex) + BaseAgeZ + C(Season) + PC1Z + PC2Z + PC3Z + PC4Z + PC5Z'\
    ' + C(BaseBMI_class) + DaysZ:C(BaseBMI_class) + DaysP2Z:C(BaseBMI_class) + DaysP3Z:C(BaseBMI_class)'
model = sm.MixedLM.from_formula(fe_formula, groups='public_client_id',
                                data=tsDF_FM.loc[tsDF_FM['BaseBMI_class']!='Underweight'],
                                re_formula='DaysZ')#Random intercepts for each group are included as default
t_start = time.time()
LMM2_FM = model.fit(method=['bfgs', 'lbfgs', 'cg', 'powell', 'nm'])#Back up for failure of convergence
t_elapsed = time.time() - t_start
print('\n• Model 2: LMM adjusted for baseline BMI-based class and general covariates')
display(LMM2_FM.summary())
print('(Note that the estimates for fixed effects and the variances for random effects are shown in the single table.)')
##Visualize b-coefs and g-coefs
tempDF = pd.DataFrame({'coef':LMM2_FM.params,
                       'coef_ci_l':LMM2_FM.conf_int(alpha=0.05).iloc[:, 0],
                       'coef_ci_h':LMM2_FM.conf_int(alpha=0.05).iloc[:, 1],
                       'pval':LMM2_FM.pvalues})
tempDF['coef_ci'] = (tempDF['coef_ci_h'] - tempDF['coef_ci_l'])/2
tempDF = tempDF.reset_index()
sns.set(style='ticks', font='Arial', context='notebook')
plt.figure(figsize=(4, 0.25*len(tempDF)))
plt.errorbar(x=tempDF['coef'], y=tempDF['index'], xerr=tempDF['coef_ci'],
             fmt='ok', ecolor='k', capsize=5)
plt.xlim(-1.0, 1.0)
sns.despine()
plt.xlabel(r'$\beta$'+' or '+r'$\gamma$'+' coefficient in LMM model\n(Mean with 95% CI)')
plt.ylabel('')
plt.axvline(x=0, **{'linestyle':'--', 'color':'k'})
for row_i in range(len(tempDF)):
    if tempDF.iloc[row_i]['pval'] < 0.05:
        plt.axhspan(ymin=row_i-0.5, ymax=row_i+0.5, facecolor='g', alpha=0.2)
plt.gca().invert_yaxis()
plt.margins(y=0.01, tight=True)
plt.show()
print('Elapsed time for fitting LMM:', round(t_elapsed//60), 'min', round(t_elapsed%60, 1), 'sec')

### 3-4. Estimated transition of biological BMI

> The days in program of the fitted values were different between individuals.  
> –> In statsmodels, MixedLMResults.predict() method use only the fixed effects parameters for prediction.  
> ***–> To evaluate the variability in the in-sample population, the rondom effects component should be added manually.***

In [ ]:
#Sex-stratified model-predicted biological BMI
print('Sex-stratified model-predicted biological BMI')

#Prepare DF to impute days_in_program but maintain in-sample population
tempA1 = tsDF_FM['public_client_id'].unique()
tempA2 = np.arange(0, 365.25/12*month_threshold + 1, 365.25/12*month_threshold/nknots)
tempL = ['Spring', 'Summer', 'Autumn', 'Winter']
tempDF = pd.DataFrame(data={'public_client_id':np.repeat(tempA1, len(tempA2)*len(tempL)),
                            'days_in_program':np.tile(np.repeat(tempA2, len(tempL)), len(tempA1)),
                            'Season':np.tile(tempL, len(tempA1)*len(tempA2))})
tempDF1 = tsDF_FM[['public_client_id', 'BaseChemBMI_class',
                   'BaseBMI_class', 'Sex', 'BaseAge', 'PC1', 'PC2', 'PC3', 'PC4', 'PC5']]
tempDF1 = tempDF1.drop_duplicates('public_client_id', keep='first')
tempDF = pd.merge(tempDF, tempDF1, on='public_client_id', how='left')
##Add dummy variables for the linear regression spline for time
tempL = [365.25/12 * month_threshold/nknots * (knot+1) for knot in range(nknots)]#Days of knots
tempL1 = []#2nd piecewise
tempL2 = []#3rd piecewise
for days in tempDF['days_in_program'].tolist():
    if days < tempL[0]:#[0 month, 6 months)
        tempL1.append(0.0)
        tempL2.append(0.0)
    elif days < tempL[1]:#[6 months, 12 months)
        tempL1.append(days - tempL[0])
        tempL2.append(0.0)
    elif days <= tempL[2]:#[12 months, 18 months]
        tempL1.append(days - tempL[0])
        tempL2.append(days - tempL[1])
tempDF['DaysP2'] = tempL1
tempDF['DaysP3'] = tempL2
##Add a dummy index to use when merging later
tempDF['KeyIndex'] = tempDF['public_client_id'] + '_day' + tempDF['days_in_program'].astype(str)
tempDF = tempDF.set_index('KeyIndex')
##Standardize numeric independent variables using the already fitted StandardScaler
tempL = ['days_in_program', 'DaysP2', 'DaysP3', 'BaseAge', 'PC1', 'PC2', 'PC3', 'PC4', 'PC5']
tempDF1 = pd.DataFrame(scaler_FM.transform(tempDF[tempL]), index=tempDF.index,
                       columns=['DaysZ', 'DaysP2Z', 'DaysP3Z', 'BaseAgeZ', 'PC1Z', 'PC2Z', 'PC3Z', 'PC4Z', 'PC5Z'])
tempDF = pd.merge(tempDF, tempDF1, left_index=True, right_index=True, how='inner')

#Model 1 prediction
##Predicted component from the fixed effects
tempS = LMM1_FM.predict(tempDF)
##Create design matrix for the random effects
tempDF1 = pd.DataFrame({'Intercept_re':np.repeat(1, len(tempDF)),
                        'Slope_re':tempDF['DaysZ']})
tempD = LMM1_FM.random_effects#Dictionary of key=group label and value=pd.Series
##Predicted component from the random effects
tempL = [np.dot(tempDF1.iloc[row_i], tempD[group_n])
         for (row_i, group_n) in enumerate(tempDF['public_client_id'].tolist())]
##Overall prediction for in-sample individuals
tempDF['ChemBMIchange_predict1'] = tempS + tempL

#Model 2 prediction
##Eliminate 'Underweight' class
tempDF1 = tempDF.loc[tempDF['BaseBMI_class']!='Underweight']
##Predicted component from the fixed effects
tempS = LMM2_FM.predict(tempDF1)
##Create design matrix for the random effects
tempDF2 = pd.DataFrame({'Intercept_re':np.repeat(1, len(tempDF1)),
                        'Slope_re':tempDF1['DaysZ']})
tempD = LMM2_FM.random_effects#Dictionary of key=group label and value=pd.Series
##Predicted component from the random effects
tempL = [np.dot(tempDF2.iloc[row_i], tempD[group_n])
         for (row_i, group_n) in enumerate(tempDF1['public_client_id'].tolist())]
##Overall prediction for in-sample individuals
tempDF1['ChemBMIchange_predict2'] = tempS + tempL
##Add prediction to the whole DF
tempDF = pd.merge(tempDF, tempDF1['ChemBMIchange_predict2'],
                  left_index=True, right_index=True, how='outer')

#Flatten effects of the dummy season variable
tempDF = tempDF.groupby(by=['public_client_id', 'days_in_program'])
tempDF = tempDF.agg({'ChemBMIchange_predict1':'mean', 'ChemBMIchange_predict2':'mean'})
tempDF = tempDF.reset_index()
##Recover the dropped covariates and save the prediction DF to use later
tempDF1 = tsDF_FM[['public_client_id', 'BaseChemBMI_class', 'BaseBMI_class', 'Sex']]
tempDF1 = tempDF1.drop_duplicates('public_client_id', keep='first')
tempDF = pd.merge(tempDF, tempDF1, on='public_client_id', how='left')
tempDF['KeyIndex'] = tempDF['public_client_id'] + '_day' + tempDF['days_in_program'].astype(str)
chemLMM_FM = tempDF.set_index('KeyIndex')
print('\n• Prediction DF')
display(chemLMM_FM)
display(chemLMM_FM.describe(include='all'))

#Plot1
print('\n• Model 1: LMM adjusted for general covariates')
sns.set(style='ticks', font='Arial', context='notebook')
plt.figure(figsize=(5, 3))
sns.lineplot(data=tsDF_FM, x='days_in_program', y='ChemBMIchange',
             units='public_client_id', estimator=None, color='gray', lw=1, alpha=0.1, legend=None)
sns.scatterplot(data=tsDF_FM, x='days_in_program', y='ChemBMIchange',
                color='gray', edgecolor='k', s=20, alpha=0.3)
p = sns.lineplot(data=chemLMM_FM, x='days_in_program', y='ChemBMIchange_predict1',
                 estimator='mean', ci=95, color='g')
p.set(xlim=(0, 365.25/12*month_threshold), xticks=np.arange(0, 365.25/12*month_threshold, 100),
      ylim=(-20, 15), yticks=np.arange(-20, 15.1, 5))
sns.despine()
plt.xlabel('Days in program')
plt.ylabel('Rate of change [%]')
plt.show()

#Plot2
print('\n• Model 2: LMM adjusted for baseline BMI-based class and general covariates')
tempD = {'Normal':'green', 'Overweight':'orange', 'Obese':'red'}
sns.set(style='ticks', font='Arial', context='notebook')
plt.figure(figsize=(5, 3))
sns.scatterplot(data=tsDF_FM[tsDF_FM['BaseBMI_class']!='Underweight'], x='days_in_program', y='ChemBMIchange',
                hue='BaseBMI_class', hue_order=tempD.keys(), palette=tempD,
                edgecolor='k', s=20, alpha=0.3)
p = sns.lineplot(data=chemLMM_FM, x='days_in_program', y='ChemBMIchange_predict2',
                 hue='BaseBMI_class', hue_order=tempD.keys(), palette=tempD,
                 estimator='mean', ci=95)
p.set(xlim=(0, 365.25/12*month_threshold), xticks=np.arange(0, 365.25/12*month_threshold, 100),
      ylim=(-20, 15), yticks=np.arange(-20, 15.1, 5))
sns.despine()
plt.xlabel('Days in program')
plt.ylabel('Rate of change [%]')
plt.legend(bbox_to_anchor=(1, 0.5), loc='center left', borderaxespad=1)
plt.show()

## 5. Measured BMI

### 5-1. Prepare the time-series DF

In [ ]:
#Import time-series BMI
fileDir = './ExportData/'
fileName = '210104_Biological-BMI-paper_data-cleaning-BMI-omics_time-series-bmiDF-without-imputation_final-cohort.tsv'
tsDF = pd.read_csv(fileDir+fileName, sep='\t', dtype={'public_client_id': str}).set_index('KeyIndex')
tsDF = tsDF.rename(columns={'BMI_CALC':'BMI'})
display(tsDF)

In [ ]:
tempDF = tsDF#Not copy just rename
tempDF['BaseBMI'] = np.e**tempDF['log_BaseBMI']

for col_n in ['BaseBMI']:
    tempL = []
    for value in tempDF[col_n].tolist():
        if np.isnan(value):
            tempL.append('NotCalculated')
        elif value < 18.5:
            tempL.append('Underweight')
        elif value < 25:
            tempL.append('Normal')
        elif value < 30:
            tempL.append('Overweight')
        elif value >= 30:
            tempL.append('Obese')
        else:#Just in case
            tempL.append('Error?')
    tempDF[col_n+'_class'] = tempL
    #Confirmation
    print(col_n+'_class:')
    tempS = tempDF[col_n+'_class'].value_counts()
    tempDF1 = pd.DataFrame({'Count':tempS, 'Percentage':tempS/len(tempDF)*100})
    display(tempDF1)
    print('')

#Clean the time-series DF
tsDF = tsDF[['public_client_id', 'days_in_program', 'Season', 'log_BMI', 'BMI',
             'log_BaseBMI', 'BaseBMI', 'BaseBMI_class',
             'Sex', 'BaseAge', 'Race', 'PC1', 'PC2', 'PC3', 'PC4', 'PC5']]
display(tsDF)

### 5-2. Prepare DFs for LMM with the linear regresion spline for time

In [ ]:
#Check measurement distribution
sns.set(style='ticks', font='Arial', context='notebook')
plt.figure(figsize=(4, 3))
sns.distplot(tsDF['days_in_program'], color='k')
sns.despine()
plt.axvline(x=365.25/12*0, **{'linestyle':'--', 'color':'gray'})
plt.axvline(x=365.25/12*6, **{'linestyle':'--', 'color':'gray'})
plt.axvline(x=365.25/12*12, **{'linestyle':'--', 'color':'gray'})
plt.axvline(x=365.25/12*18, **{'linestyle':'--', 'color':'gray'})
plt.axvspan(xmin=365.25/12*18, xmax=plt.gca().get_xlim()[1], facecolor='gray', alpha=0.2)
plt.ylabel('Density')
plt.xlabel('Days in program')
plt.show()

In [ ]:
#Drop unnecessary variables
tempDF = tsDF[['public_client_id', 'days_in_program', 'Season', 'BMI', 'BaseBMI',
               'BaseBMI_class', 'Sex', 'BaseAge', 'PC1', 'PC2', 'PC3', 'PC4', 'PC5']]
print('Time-series DF nrows before elimination:', len(tempDF))
print(' -> Unique ID:', len(tempDF['public_client_id'].unique()))

#Select measurements used in the longitudinal analysis
month_threshold = 18
tempDF = tempDF.loc[tempDF['days_in_program'] <= 365.25/12*month_threshold]
print('Time-series DF nrows after selecting measurements during the taget period:', len(tempDF))
print(' -> Unique ID:', len(tempDF['public_client_id'].unique()))

#Select the cohort used in the longitudinal analysis
##Select participants who have more than 2 measuremnts in BMI
fileDir = './ExportData/'
fileName = '210104_Biological-BMI-paper_data-cleaning-BMI-omics_time-series-bmiDF-without-imputation_final-cohort.tsv'
tempDF1 = pd.read_csv(fileDir+fileName, sep='\t', dtype={'public_client_id': str}).set_index('public_client_id')
tempDF1 = tempDF1.loc[tempDF1['days_in_program'] <= 365.25/12*month_threshold]
tempS = tempDF1.index.value_counts()
tempL1 = tempS.loc[tempS>=2].index.tolist()
##Select participants who have more than 2 measuremnts in omics
fileDir = './ExportData/'
fileName = '210104_Biological-BMI-paper_data-cleaning-BMI-omics_time-series-combiDF-without-imputation_final-cohort.tsv'
tempDF1 = pd.read_csv(fileDir+fileName, sep='\t', dtype={'public_client_id': str}).set_index('public_client_id')
tempDF1 = tempDF1.loc[tempDF1['days_in_program'] <= 365.25/12*month_threshold]
tempS = tempDF1.index.value_counts()
tempL2 = tempS.loc[tempS>=2].index.tolist()
##Select participants who have more than 2 measuremnts in both BMI and omics
tempL = list(set(tempL1) & set(tempL2))
print('Participants who have more than 2 measurements in both BMI and omics:', len(tempL))
tempDF = tempDF.loc[tempDF['public_client_id'].isin(tempL)]
print('Time-series DF nrows after selecting participants who have ≥2 measuremnts in both BMI and omics:',
      len(tempDF))
print(' -> Unique ID:', len(tempDF['public_client_id'].unique()))

#Select the participants who have all covariates
tempDF = tempDF.dropna()
print('Time-series DF nrows after selecting participants who have all covariates:', len(tempDF))
print(' -> Unique ID:', len(tempDF['public_client_id'].unique()))

#Check measurement distribution
sns.set(style='ticks', font='Arial', context='talk')
plt.figure(figsize=(4, 3))
sns.distplot(tsDF['days_in_program'], color='k')
sns.despine()
plt.axvline(x=365.25/12*0, **{'linestyle':'--', 'color':'gray'})
plt.axvline(x=365.25/12*6, **{'linestyle':'--', 'color':'gray'})
plt.axvline(x=365.25/12*12, **{'linestyle':'--', 'color':'gray'})
plt.axvline(x=365.25/12*18, **{'linestyle':'--', 'color':'gray'})
plt.axvspan(xmin=365.25/12*18, xmax=plt.gca().get_xlim()[1], facecolor='gray', alpha=0.2)
plt.ylabel('Density')
plt.xlabel('Days in program')
plt.show()

#Calculate rate of change
tempDF['BMIchange'] = (tempDF['BMI'] - tempDF['BaseBMI']) / tempDF['BaseBMI'] * 100

#Check distribution
print('Skewness of rate of change:', stats.skew(tempDF['BMIchange']))
sns.set(style='ticks', font='Arial', context='notebook')
plt.figure(figsize=(4, 3))
sns.distplot(tempDF['BMIchange'], color='k')
sns.despine()
plt.ylabel('Density')
plt.xlabel('Rate of change [%]')
plt.show()

#Add dummy variables for the linear regression spline for time
nknots = 3
tempL = [365.25/12 * month_threshold/nknots * (knot+1) for knot in range(nknots)]#Days of knots
tempL1 = []#2nd piece
tempL2 = []#3rd piece
for days in tempDF['days_in_program'].tolist():
    if days < tempL[0]:#[0 month, 6 months)
        tempL1.append(0.0)
        tempL2.append(0.0)
    elif days < tempL[1]:#[6 months, 12 months)
        tempL1.append(days - tempL[0])
        tempL2.append(0.0)
    elif days <= tempL[2]:#[12 months, 18 months]
        tempL1.append(days - tempL[0])
        tempL2.append(days - tempL[1])
tempDF['DaysP2'] = tempL1
tempDF['DaysP3'] = tempL2

#Sort to make 'Normal' the standard in the baseline BMI-stratified LMM (Normal < Obese < Overweight)
tempDF = tempDF.sort_values(by=['BaseBMI_class'], ascending=True)

#Standardize numeric independent variables
scaler_BS = StandardScaler(copy=True, with_mean=True, with_std=True)#Use later again during predictions
tempL = ['days_in_program', 'DaysP2', 'DaysP3', 'BaseAge', 'PC1', 'PC2', 'PC3', 'PC4', 'PC5']
scaler_BS.fit(tempDF[tempL])
tempDF1 = pd.DataFrame(scaler_BS.transform(tempDF[tempL]), index=tempDF.index,
                       columns=['DaysZ', 'DaysP2Z', 'DaysP3Z', 'BaseAgeZ', 'PC1Z', 'PC2Z', 'PC3Z', 'PC4Z', 'PC5Z'])
tempDF = pd.merge(tempDF, tempDF1, left_index=True, right_index=True, how='inner')

#Confirm examples
sns.set(style='ticks', font='Arial', context='notebook')
plt.figure(figsize=(4, 3))
sns.distplot(tempDF['DaysZ'], label='DaysZ')
sns.distplot(tempDF['BaseAgeZ'], label='BaseAgeZ')
sns.distplot(tempDF['PC1Z'], label='PC1Z')
sns.despine()
plt.ylabel('Density')
plt.xlabel('Z-score')
tempL = [365.25/12 * month_threshold/nknots * (knot+1) for knot in range(nknots)]#Days of knots
tempS = (pd.Series(tempL) - scaler_BS.mean_[0]) / scaler_BS.scale_[0]#Z-score transformation
for knot in tempS:
    plt.axvline(x=knot, **{'linestyle':'--', 'color':'gray'})
plt.legend(bbox_to_anchor=(1, 0.5), loc='center left', borderaxespad=1)
plt.show()

#Update
tsDF = tempDF
display(tsDF)

### 5-3. LMM with random intercepts and random slopes for time

> Days in the program is fitted as a numeric variable using linear regression splines (cf. Zubair, N. et al. Sci. Rep. 2019).  
> ***–> Allows differences in the trajectory of BMI change.***
>
> Random intercepts and random slopes for days in the program are used as random effects (cf. Zubair, N. et al. Sci. Rep. 2019).  
> ***–> Allows individual differences in the intercept and the rate of change of BMI over the program.***  
> –> Random slope is applied not for the piecewise variable of time but only for the overall variable of time; otherwise, leads to "LinAlgError: Singular matrix" due to small sample size per most individuals. Also, it is not so weird to assume the random slope is consistent during the program.  
>
> ***Eliminate "Underweight" class in the baseline obesity class-stratified models.***  
> <– Otherwise, leads to "LinAlgError: Singular matrix" due to small sample population.  

In [ ]:
#Overall LMM adjusted for general covariates
fe_formula = 'BMIchange ~ DaysZ + DaysP2Z + DaysP3Z'\
    ' + C(Sex) + BaseAgeZ + C(Season) + PC1Z + PC2Z + PC3Z + PC4Z + PC5Z'
model = sm.MixedLM.from_formula(fe_formula, groups='public_client_id', data=tsDF,
                                re_formula='DaysZ')#Random intercepts for each group are included as default
t_start = time.time()
LMM1_BS = model.fit(method=['bfgs', 'lbfgs', 'cg', 'powell', 'nm'])#Back up for failure of convergence
t_elapsed = time.time() - t_start
print('\n• Model 1: LMM adjusted for general covariates')
display(LMM1_BS.summary())
print('(Note that the estimates for fixed effects and the variances for random effects are shown in the single table.)')
##Visualize b-coefs and g-coefs
tempDF = pd.DataFrame({'coef':LMM1_BS.params,
                       'coef_ci_l':LMM1_BS.conf_int(alpha=0.05).iloc[:, 0],
                       'coef_ci_h':LMM1_BS.conf_int(alpha=0.05).iloc[:, 1],
                       'pval':LMM1_BS.pvalues})
tempDF['coef_ci'] = (tempDF['coef_ci_h'] - tempDF['coef_ci_l'])/2
tempDF = tempDF.reset_index()
sns.set(style='ticks', font='Arial', context='notebook')
plt.figure(figsize=(4, 0.25*len(tempDF)))
plt.errorbar(x=tempDF['coef'], y=tempDF['index'], xerr=tempDF['coef_ci'],
             fmt='ok', ecolor='k', capsize=5)
plt.xlim(-1.0, 1.0)
sns.despine()
plt.xlabel(r'$\beta$'+' or '+r'$\gamma$'+' coefficient in LMM model\n(Mean with 95% CI)')
plt.ylabel('')
plt.axvline(x=0, **{'linestyle':'--', 'color':'k'})
for row_i in range(len(tempDF)):
    if tempDF.iloc[row_i]['pval'] < 0.05:
        plt.axhspan(ymin=row_i-0.5, ymax=row_i+0.5, facecolor='k', alpha=0.2)
plt.gca().invert_yaxis()
plt.margins(y=0.01, tight=True)
plt.show()
print('Elapsed time for fitting LMM:', round(t_elapsed//60), 'min', round(t_elapsed%60, 1), 'sec')

#Baseline BMI-stratified LMM adjusted for general covariates
fe_formula = 'BMIchange ~ DaysZ + DaysP2Z + DaysP3Z'\
    ' + C(Sex) + BaseAgeZ + C(Season) + PC1Z + PC2Z + PC3Z + PC4Z + PC5Z'\
    ' + C(BaseBMI_class) + DaysZ:C(BaseBMI_class) + DaysP2Z:C(BaseBMI_class) + DaysP3Z:C(BaseBMI_class)'
model = sm.MixedLM.from_formula(fe_formula, groups='public_client_id',
                                data=tsDF.loc[tsDF['BaseBMI_class']!='Underweight'],
                                re_formula='DaysZ')#Random intercepts for each group are included as default
t_start = time.time()
LMM2_BS = model.fit(method=['bfgs', 'lbfgs', 'cg', 'powell', 'nm'])#Back up for failure of convergence
t_elapsed = time.time() - t_start
print('\n• Model 2: LMM adjusted for baseline BMI-based class and general covariates')
display(LMM2_BS.summary())
print('(Note that the estimates for fixed effects and the variances for random effects are shown in the single table.)')
##Visualize b-coefs and g-coefs
tempDF = pd.DataFrame({'coef':LMM2_BS.params,
                       'coef_ci_l':LMM2_BS.conf_int(alpha=0.05).iloc[:, 0],
                       'coef_ci_h':LMM2_BS.conf_int(alpha=0.05).iloc[:, 1],
                       'pval':LMM2_BS.pvalues})
tempDF['coef_ci'] = (tempDF['coef_ci_h'] - tempDF['coef_ci_l'])/2
tempDF = tempDF.reset_index()
sns.set(style='ticks', font='Arial', context='notebook')
plt.figure(figsize=(4, 0.25*len(tempDF)))
plt.errorbar(x=tempDF['coef'], y=tempDF['index'], xerr=tempDF['coef_ci'],
             fmt='ok', ecolor='k', capsize=5)
plt.xlim(-1.0, 1.0)
sns.despine()
plt.xlabel(r'$\beta$'+' or '+r'$\gamma$'+' coefficient in LMM model\n(Mean with 95% CI)')
plt.ylabel('')
plt.axvline(x=0, **{'linestyle':'--', 'color':'k'})
for row_i in range(len(tempDF)):
    if tempDF.iloc[row_i]['pval'] < 0.05:
        plt.axhspan(ymin=row_i-0.5, ymax=row_i+0.5, facecolor='k', alpha=0.2)
plt.gca().invert_yaxis()
plt.margins(y=0.01, tight=True)
plt.show()
print('Elapsed time for fitting LMM:', round(t_elapsed//60), 'min', round(t_elapsed%60, 1), 'sec')

### 5-4. Estimated transition of BMI

> The days in program of the fitted values were different between individuals.  
> –> In statsmodels, MixedLMResults.predict() method use only the fixed effects parameters for prediction.  
> ***–> To evaluate the variability in the in-sample population, the rondom effects component should be added manually.***

In [ ]:
#Prepare DF to impute days_in_program but maintain in-sample population
tempA1 = tsDF['public_client_id'].unique()
tempA2 = np.arange(0, 365.25/12*month_threshold + 1, 365.25/12*month_threshold/nknots)
tempL = ['Spring', 'Summer', 'Autumn', 'Winter']
tempDF = pd.DataFrame(data={'public_client_id':np.repeat(tempA1, len(tempA2)*len(tempL)),
                            'days_in_program':np.tile(np.repeat(tempA2, len(tempL)), len(tempA1)),
                            'Season':np.tile(tempL, len(tempA1)*len(tempA2))})
tempDF1 = tsDF[['public_client_id', 'BaseBMI_class', 'Sex', 'BaseAge', 'PC1', 'PC2', 'PC3', 'PC4', 'PC5']]
tempDF1 = tempDF1.drop_duplicates('public_client_id', keep='first')
tempDF = pd.merge(tempDF, tempDF1, on='public_client_id', how='left')
##Add dummy variables for the linear regression spline for time
tempL = [365.25/12 * month_threshold/nknots * (knot+1) for knot in range(nknots)]#Days of knots
tempL1 = []#2nd piecewise
tempL2 = []#3rd piecewise
for days in tempDF['days_in_program'].tolist():
    if days < tempL[0]:#[0 month, 6 months)
        tempL1.append(0.0)
        tempL2.append(0.0)
    elif days < tempL[1]:#[6 months, 12 months)
        tempL1.append(days - tempL[0])
        tempL2.append(0.0)
    elif days <= tempL[2]:#[12 months, 18 months]
        tempL1.append(days - tempL[0])
        tempL2.append(days - tempL[1])
tempDF['DaysP2'] = tempL1
tempDF['DaysP3'] = tempL2
##Add a dummy index to use when merging later
tempDF['KeyIndex'] = tempDF['public_client_id'] + '_day' + tempDF['days_in_program'].astype(str)
tempDF = tempDF.set_index('KeyIndex')
##Standardize numeric independent variables using the already fitted StandardScaler
tempL = ['days_in_program', 'DaysP2', 'DaysP3', 'BaseAge', 'PC1', 'PC2', 'PC3', 'PC4', 'PC5']
tempDF1 = pd.DataFrame(scaler_BS.transform(tempDF[tempL]), index=tempDF.index,
                       columns=['DaysZ', 'DaysP2Z', 'DaysP3Z', 'BaseAgeZ', 'PC1Z', 'PC2Z', 'PC3Z', 'PC4Z', 'PC5Z'])
tempDF = pd.merge(tempDF, tempDF1, left_index=True, right_index=True, how='inner')

#Model 1 prediction
##Predicted component from the fixed effects
tempS = LMM1_BS.predict(tempDF)
##Create design matrix for the random effects
tempDF1 = pd.DataFrame({'Intercept_re':np.repeat(1, len(tempDF)),
                        'Slope_re':tempDF['DaysZ']})
tempD = LMM1_BS.random_effects#Dictionary of key=group label and value=pd.Series
##Predicted component from the random effects
tempL = [np.dot(tempDF1.iloc[row_i], tempD[group_n])
         for (row_i, group_n) in enumerate(tempDF['public_client_id'].tolist())]
##Overall prediction for in-sample individuals
tempDF['BMIchange_predict1'] = tempS + tempL

#Model 2 prediction
##Eliminate 'Underweight' class
tempDF1 = tempDF.loc[tempDF['BaseBMI_class']!='Underweight']
##Predicted component from the fixed effects
tempS = LMM2_BS.predict(tempDF1)
##Create design matrix for the random effects
tempDF2 = pd.DataFrame({'Intercept_re':np.repeat(1, len(tempDF1)),
                        'Slope_re':tempDF1['DaysZ']})
tempD = LMM2_BS.random_effects#Dictionary of key=group label and value=pd.Series
##Predicted component from the random effects
tempL = [np.dot(tempDF2.iloc[row_i], tempD[group_n])
         for (row_i, group_n) in enumerate(tempDF1['public_client_id'].tolist())]
##Overall prediction for in-sample individuals
tempDF1['BMIchange_predict2'] = tempS + tempL
##Add prediction to the whole DF
tempDF = pd.merge(tempDF, tempDF1['BMIchange_predict2'],
                  left_index=True, right_index=True, how='outer')

#Flatten effects of the dummy season variable
tempDF = tempDF.groupby(by=['public_client_id', 'days_in_program'])
tempDF = tempDF.agg({'BMIchange_predict1':'mean', 'BMIchange_predict2':'mean'})
tempDF = tempDF.reset_index()
##Recover the dropped covariates and save the prediction DF to use later
tempDF1 = tsDF[['public_client_id', 'BaseBMI_class', 'Sex']]
tempDF1 = tempDF1.drop_duplicates('public_client_id', keep='first')
tempDF = pd.merge(tempDF, tempDF1, on='public_client_id', how='left')
tempDF['KeyIndex'] = tempDF['public_client_id'] + '_day' + tempDF['days_in_program'].astype(str)
measLMM = tempDF.set_index('KeyIndex')
print('\n• Prediction DF')
display(measLMM)
display(measLMM.describe(include='all'))

#Plot1
print('\n• Model 1: LMM adjusted for general covariates')
sns.set(style='ticks', font='Arial', context='notebook')
plt.figure(figsize=(5, 3))
sns.lineplot(data=tsDF, x='days_in_program', y='BMIchange',
             units='public_client_id', estimator=None, color='gray', lw=1, alpha=0.1, legend=None)
sns.scatterplot(data=tsDF, x='days_in_program', y='BMIchange',
                color='gray', edgecolor='k', s=20, alpha=0.3)
p = sns.lineplot(data=measLMM, x='days_in_program', y='BMIchange_predict1',
                 estimator='mean', ci=95, color='k')
p.set(xlim=(0, 365.25/12*month_threshold), xticks=np.arange(0, 365.25/12*month_threshold, 100),
      ylim=(-20, 15), yticks=np.arange(-20, 15.1, 5))
sns.despine()
plt.xlabel('Days in program')
plt.ylabel('Rate of change [%]')
plt.show()

#Plot2
print('\n• Model 2: LMM adjusted for baseline BMI-based class and general covariates')
tempD = {'Normal':'green', 'Overweight':'orange', 'Obese':'red'}
sns.set(style='ticks', font='Arial', context='notebook')
plt.figure(figsize=(5, 3))
sns.scatterplot(data=tsDF[tsDF['BaseBMI_class']!='Underweight'], x='days_in_program', y='BMIchange',
                hue='BaseBMI_class', hue_order=tempD.keys(), palette=tempD,
                edgecolor='k', s=20, alpha=0.3)
p = sns.lineplot(data=measLMM, x='days_in_program', y='BMIchange_predict2',
                 hue='BaseBMI_class', hue_order=tempD.keys(), palette=tempD,
                 estimator='mean', ci=95)
p.set(xlim=(0, 365.25/12*month_threshold), xticks=np.arange(0, 365.25/12*month_threshold, 100),
      ylim=(-20, 15), yticks=np.arange(-20, 15.1, 5))
sns.despine()
plt.xlabel('Days in program')
plt.ylabel('Rate of change [%]')
plt.legend(bbox_to_anchor=(1, 0.5), loc='center left', borderaxespad=1)
plt.show()

## 6. Comparison b/w BMI types

### 6-1. Overall and the baseline BMI class stratification

In [ ]:
#For presentation
print('For presentation: sex-stratified model-predicted biological BMI')

#Merge prediction DFs
tempDF = measLMM.drop(columns=['Sex'])
tempL = [metLMM_FM, protLMM_FM, chemLMM_FM]
for df_n in tempL:
    tempDF1 = df_n.drop(columns=['public_client_id', 'days_in_program', 'BaseBMI_class', 'Sex'])
    tempDF = pd.merge(tempDF, tempDF1, left_index=True, right_index=True, how='outer')
tempDF = pd.merge(tempDF, measLMM['Sex'], left_index=True, right_index=True)

#Model 1
print('\n• Model 1: LMM adjusted for general covariates')
tempD = {'BMI':'k', 'MetBMI':'b', 'ProtBMI':'r', 'ChemBMI':'g'}
tempL = []
for bmi in tempD.keys():
    count = len(tempDF.loc[~tempDF[bmi+'change_predict1'].isnull()]['public_client_id'].unique())
    tempL.append(bmi+' (n = '+str(count)+')')
print('  -> Confirm each sample size:', tempL)
tempL = tempDF.loc[:, tempDF.columns.str.contains('change_predict1', regex=True)].columns.tolist()
tempDF1 = tempDF.reset_index().melt(var_name='LMM', value_name='PredictedChange', value_vars=tempL,
                                    id_vars=['public_client_id', 'days_in_program', 'BaseBMI_class'])
tempDF1 = tempDF1.dropna()
tempDF1['LMM'] = tempDF1['LMM'].str.replace('change_predict1', '')
sns.set(style='ticks', font='Arial', context='talk')
plt.figure(figsize=(5, 4))
p = sns.lineplot(data=tempDF1, x='days_in_program', y='PredictedChange',
                 hue='LMM', hue_order=tempD.keys(), palette=tempD, estimator='mean', ci=95)
p.set(xlim=(0, 365.25/12*12), xticks=np.arange(0, 365.25/12*12+1, 90),
      ylim=(-8.5, 2.5), yticks=np.arange(-8, 2.5, 2))
sns.despine()
plt.xlabel('Days in program')
plt.ylabel('Estimated rate of change [%]')
plt.axhline(y=0, **{'linestyle':'--', 'color':'k'}, zorder=0)
for knot in range(nknots-1):
    if knot%2 == 1:
        plt.axvspan(xmin=365.25/12*month_threshold/nknots*knot,
                    xmax=365.25/12*month_threshold/nknots*(knot+1),
                    facecolor='gray', alpha=0.2, zorder=-1)
#Generate standard Matplotlib legend manually
tempL = []
for bmi in tempD.keys():
    tempL.append(mlines.Line2D([], [], color=tempD[bmi], label=bmi, linewidth=3))
plt.legend(handles=tempL, fontsize='medium', title='LMM estimate', title_fontsize='large',
           bbox_to_anchor=(0.5, 0), loc='upper center', ncol=2, borderaxespad=4, frameon=True)
##Save
fileDir = './ExportFigures/'
fileName = '210823_Biological-BMI-paper_longitudinal-LMM-ver3_Overall.tif'
plt.gcf().savefig(fileDir+fileName, dpi=300, bbox_inches='tight', pad_inches=0,
                  pil_kwargs={'compression':'tiff_lzw'})
plt.show()

#Model 2
print('\n• Model 2: LMM adjusted for baseline BMI-based class and general covariates')
tempD = {'BMI':'k', 'MetBMI':'b', 'ProtBMI':'r', 'ChemBMI':'g'}
tempL1 = ['Normal', 'Overweight', 'Obese']
for bmi_class in tempL1:
    print('   - ' + bmi_class + ' class based on the baseline BMI')
    tempDF1 = tempDF.loc[tempDF['BaseBMI_class']==bmi_class]
    tempL = []
    for bmi in tempD.keys():
        count = len(tempDF1.loc[~tempDF1[bmi+'change_predict2'].isnull()]['public_client_id'].unique())
        tempL.append(bmi+' (n = '+str(count)+')')
    print('     -> Confirm each sample size:', tempL)
tempDF1 = tempDF.loc[tempDF['BaseBMI_class'].isin(tempL1)]
tempL = tempDF1.loc[:, tempDF1.columns.str.contains('change_predict2', regex=True)].columns.tolist()
tempDF1 = tempDF1.reset_index().melt(var_name='LMM', value_name='PredictedChange', value_vars=tempL,
                                     id_vars=['public_client_id', 'days_in_program', 'BaseBMI_class'])
tempDF1 = tempDF1.dropna()
tempDF1['LMM'] = tempDF1['LMM'].str.replace('change_predict2', '')
sns.set(style='ticks', font='Arial', context='talk')
p = sns.relplot(kind='line', data=tempDF1, x='days_in_program', y='PredictedChange',
                row='BaseBMI_class', row_order=tempL1,
                hue='LMM', hue_order=tempD.keys(), palette=tempD, estimator='mean', ci=95,
                height=4, aspect=5/4+0.075, legend=False)
p.set(xlim=(0, 365.25/12*12), xticks=np.arange(0, 365.25/12*12+1, 90),
      ylim=(-8.5, 2.5), yticks=np.arange(-8, 2.5, 2))
tempL = ['Baseline class: '+bmi_class for bmi_class in tempL1]
for ax, bmi_class in zip(p.axes.ravel(), tempL):
    #Draw initial line
    ax.axhline(y=0, **{'linestyle':'--', 'color':'k'}, zorder=0)
    #Draw span b/w knots
    for knot in range(nknots-1):
        if knot%2 == 1:
            ax.axvspan(xmin=365.25/12*month_threshold/nknots*knot,
                       xmax=365.25/12*month_threshold/nknots*(knot+1),
                       facecolor='gray', alpha=0.2, zorder=-1)
    #Change facet label
    ax.set_title(bmi_class, {'fontsize':'large'})
#Reset and generate common axis title
for row_i, ax in enumerate(p.axes.ravel()):
    if row_i == 2:
        ax.set_xlabel('Days in program')
        ax.set_ylabel('')
    elif row_i == 1:
        ax.set_xlabel('')
        ax.set_ylabel('Estimated rate of change [%]')
    else:
        ax.set_xlabel('')
        ax.set_ylabel('')
plt.tight_layout()
#Generate standard Matplotlib legend manually
tempL = []
for bmi in tempD.keys():
    tempL.append(mlines.Line2D([], [], color=tempD[bmi], label=bmi, linewidth=3))
plt.legend(handles=tempL, fontsize='medium', title='LMM estimate', title_fontsize='large',
           bbox_to_anchor=(0.5, 0), loc='upper center', ncol=2, borderaxespad=4, frameon=True)
##Save
fileDir = './ExportFigures/'
fileName = '210823_Biological-BMI-paper_longitudinal-LMM-ver3_baselineBMIclass.tif'
plt.gcf().savefig(fileDir+fileName, dpi=300, bbox_inches='tight', pad_inches=0,
                  pil_kwargs={'compression':'tiff_lzw'})
plt.show()

### 6-2. Misclassification

In [ ]:
#For presentation 3
print('For presentation 3: sex-stratified model-predicted biological BMI')

#Merge prediction DFs
tempDF = measLMM.drop(columns=['Sex'])
tempL = [metLMM_FM, protLMM_FM, chemLMM_FM]
for df_n in tempL:
    tempDF1 = df_n.drop(columns=['public_client_id', 'days_in_program', 'BaseBMI_class', 'Sex'])
    tempDF = pd.merge(tempDF, tempDF1, left_index=True, right_index=True, how='outer')
tempDF = pd.merge(tempDF, measLMM['Sex'], left_index=True, right_index=True)

#Model 2
print('\n• Model 2: LMM adjusted for baseline BMI-based class and general covariates')
tempD = {'BMI':'k', 'MetBMI':'b'}
tempD1 = {'Matched':plt.get_cmap('tab10')(0), 'Misclassified':plt.get_cmap('tab10')(1)}
tempL1 = ['Normal', 'Obese']
for bmi_class in tempL1:
    print('   - ' + bmi_class + ' class based on the baseline BMI')
    for bmi in tempD.keys():
        if bmi != 'BMI':
            tempDF1 = tempDF.loc[tempDF['BaseBMI_class']==bmi_class]
            #Misclassification
            print('     -> Misclassification:')
            tempL = []
            for bbmi_class in tempDF1['Base'+bmi+'_class'].tolist():
                if bbmi_class == bmi_class:
                    tempL.append('Matched')
                else:
                    tempL.append('Misclassified')
            tempDF1['vs_Base'+bmi+'_class'] = tempL
            tempDF2 = tempDF1.drop_duplicates('public_client_id', keep='first')
            display(tempDF2['vs_Base'+bmi+'_class'].value_counts())
            tempL = ['BMIchange_predict2', bmi+'change_predict2']
            tempDF1 = tempDF1.reset_index().melt(var_name='LMM', value_name='PredictedChange', value_vars=tempL,
                                                 id_vars=['public_client_id', 'days_in_program', 'vs_Base'+bmi+'_class'])
            tempDF1 = tempDF1.dropna()
            tempDF1['LMM'] = tempDF1['LMM'].str.replace('change_predict2', '')
            sns.set(style='ticks', font='Arial', context='talk')
            p = sns.relplot(kind='line', data=tempDF1, x='days_in_program', y='PredictedChange',
                            row='LMM', row_order=['BMI', bmi],
                            hue='vs_Base'+bmi+'_class', hue_order=tempD1.keys(), palette=tempD1,
                            estimator='mean', ci=95, height=4, aspect=5/4+0.075, legend=False)
            if bmi_class=='Normal':
                p.set(xlim=(0, 365.25/12*12), xticks=np.arange(0, 365.25/12*12+1, 90),
                      ylim=(-7, 5), yticks=np.arange(-6, 5, 2))
            elif bmi_class=='Obese':
                p.set(xlim=(0, 365.25/12*12), xticks=np.arange(0, 365.25/12*12+1, 90),
                      ylim=(-8.5, 2.5), yticks=np.arange(-8, 2.5, 2))
            tempL = ['LMM estimate: '+label for label in ['BMI', bmi]]
            for ax, label in zip(p.axes.ravel(), tempL):
                #Draw initial line
                ax.axhline(y=0, **{'linestyle':'--', 'color':'k'}, zorder=0)
                #Draw span b/w knots
                for knot in range(nknots-1):
                    if knot%2 == 1:
                        ax.axvspan(xmin=365.25/12*month_threshold/nknots*knot,
                                   xmax=365.25/12*month_threshold/nknots*(knot+1),
                                   facecolor='gray', alpha=0.2, zorder=-1)
                #Change facet label
                ax.set_title(label, {'fontsize':'large'})
            #Reset and generate common axis title
            for row_i, ax in enumerate(p.axes.ravel()):
                if row_i == 1:
                    ax.set_xlabel('Days in program')
                    ax.set_ylabel('')
                else:
                    ax.set_xlabel('')
                    ax.set_ylabel('')
            p.fig.text(x=0, y=0.5, s='Estimated rate of change [%]', fontsize='medium',
                       verticalalignment='center', horizontalalignment='left', rotation='vertical')
            plt.tight_layout()
            #Generate standard Matplotlib legend manually
            tempL = []
            for misclass in tempD1.keys():
                tempL.append(mlines.Line2D([], [], color=tempD1[misclass], label=misclass, linewidth=3))
            plt.legend(handles=tempL, fontsize='medium',
                       title='Baseline class (vs. Metabolomics)', title_fontsize='large',
                       bbox_to_anchor=(0.5, 0), loc='upper center', ncol=2, borderaxespad=4, frameon=True)
            ##Save
            fileDir = './ExportFigures/'
            fileName = '210823_Biological-BMI-paper_longitudinal-LMM-ver3_misclassification-'+bmi_class+'-'+bmi+'.tif'
            plt.gcf().savefig(fileDir+fileName, dpi=300, bbox_inches='tight', pad_inches=0,
                              pil_kwargs={'compression':'tiff_lzw'})
            plt.show()